# Baby Step 4 - Tax Planning Exo-Brain
## First governed tax-committee product

This notebook preserves Steps 0-3 and creates a Step 4 copy containing the portfolio memorandum, dashboard, exhibits, decision log, and bounded committee decisions. All content is synthetic and no tax structure or implementation is authorized.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Committee control architecture

Step 4 converts the analytical record into a decision product without approving any structure. Robust cases enter standard diligence; managed-fragility cases enter enhanced diligence with mitigations; the fragile Cobalt case remains excluded pending redesign.


In [ ]:
import csv
import hashlib
import json
import shutil
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path

STEP4_QUARTER = "2026-Q3"
STEP4_COMMITTEE_VERSION = "2026-Q3-C001"
STEP4_DATE = date(2026, 7, 21).isoformat()


def read_csv(path: Path) -> list[dict]:
    with path.open(encoding="utf-8", newline="") as stream:
        return list(csv.DictReader(stream))


def write_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        raise ValueError(f"No rows supplied for {path}")
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.rstrip() + "\n", encoding="utf-8")


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def bool_text(value) -> str:
    return str(value).strip().lower()


def md_table(rows: list[dict], columns: list[tuple[str, str]]) -> str:
    header = "| " + " | ".join(label for _, label in columns) + " |"
    rule = "|" + "|".join("---" for _ in columns) + "|"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(key, "")) for key, _ in columns) + " |")
    return "\n".join([header, rule, *body])


def committee_route(resilience_class: str) -> tuple[str, str, str, str]:
    if resilience_class == "ROBUST":
        return (
            "ADVANCE_BOUNDED_DILIGENCE",
            "AUTHORIZE_STEP_5_DILIGENCE",
            "STANDARD",
            "Committee may commission synthetic diligence on the four unresolved tensions; no structure is approved.",
        )
    if resilience_class == "MANAGED_FRAGILITY":
        return (
            "ADVANCE_DILIGENCE_WITH_MITIGATIONS",
            "AUTHORIZE_STEP_5_ENHANCED_DILIGENCE",
            "ENHANCED",
            "Committee may commission synthetic diligence only with a documented mitigation plan and exception owner.",
        )
    return (
        "HOLD_FOR_REDESIGN",
        "REOPEN_AND_REDESIGN_BEFORE_DILIGENCE",
        "EXCLUDED",
        "Recommendation is excluded from the Step 5 diligence set until a substance-led redesign is documented.",
    )


def apply_step4(source_vault: Path, output_vault: Path) -> dict:
    """Copy Step 3 and create the first bounded synthetic tax-committee product."""
    source_vault = Path(source_vault)
    output_vault = Path(output_vault)
    required = [
        source_vault / "13_Audit" / "STEP_3_SUCCESS.md",
        source_vault / "16_Data" / "tax_codes.csv",
        source_vault / "16_Data" / "conglomerates.csv",
        source_vault / "16_Data" / "recommendations.csv",
        source_vault / "16_Data" / "recommendation_resilience.csv",
        source_vault / "16_Data" / "recommendation_traceability_summary.csv",
        source_vault / "16_Data" / "contradictions.csv",
        source_vault / "16_Data" / "atomic_tax_claims.csv",
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Step 3 source is incomplete:\n" + "\n".join(missing))

    if output_vault.exists():
        shutil.rmtree(output_vault)
    shutil.copytree(source_vault, output_vault)

    data = output_vault / "16_Data"
    tax_codes = read_csv(data / "tax_codes.csv")
    conglomerates = read_csv(data / "conglomerates.csv")
    recommendations = read_csv(data / "recommendations.csv")
    resilience = read_csv(data / "recommendation_resilience.csv")
    traceability = read_csv(data / "recommendation_traceability_summary.csv")
    contradictions = read_csv(data / "contradictions.csv")
    claims = read_csv(data / "atomic_tax_claims.csv")
    entities = read_csv(data / "entities.csv")
    transactions = read_csv(data / "intercompany_transactions.csv")
    sources = read_csv(data / "source_provenance.csv")

    if not (len(tax_codes) == 100 and len(conglomerates) == 10 and len(recommendations) == 10):
        raise AssertionError("Step 4 requires 100 codes, 10 conglomerates, and 10 recommendations.")
    if not (len(sources) == 44 and len(claims) == 70 and len(contradictions) == 40):
        raise AssertionError("Step 4 requires the validated Step 3 provenance state.")

    res_by_rec = {row["recommendation_id"]: row for row in resilience}
    trace_by_rec = {row["recommendation_id"]: row for row in traceability}
    contradictions_by_rec: defaultdict[str, list[dict]] = defaultdict(list)
    for row in contradictions:
        contradictions_by_rec[row["recommendation_id"]].append(row)

    decision_rows: list[dict] = []
    portfolio_rows: list[dict] = []
    condition_rows: list[dict] = []
    exhibit_rows: list[dict] = []
    decision_by_rec: dict[str, dict] = {}

    for index, rec in enumerate(recommendations, 1):
        rec_id = rec["recommendation_id"]
        res = res_by_rec[rec_id]
        trace = trace_by_rec[rec_id]
        route, permitted_action, diligence_tier, rationale = committee_route(res["resilience_class"])
        committee_decision_id = f"DEC-{index:03d}-C001"
        decision = {
            "committee_decision_id": committee_decision_id,
            "prior_decision_gate_id": rec["decision_id"],
            "recommendation_id": rec_id,
            "conglomerate_id": rec["conglomerate_id"],
            "conglomerate": rec["conglomerate"],
            "committee_version": STEP4_COMMITTEE_VERSION,
            "committee_route": route,
            "permitted_internal_action": permitted_action,
            "diligence_tier": diligence_tier,
            "committee_rationale": rationale,
            "recommendation_approved": False,
            "implementation_authorized": False,
            "synthetic": True,
        }
        decision_rows.append(decision)
        decision_by_rec[rec_id] = decision

        portfolio_rows.append({
            "recommendation_id": rec_id,
            "conglomerate_id": rec["conglomerate_id"],
            "conglomerate": rec["conglomerate"],
            "design_pattern": rec["design_pattern"],
            "candidate_hub": rec["selected_hub_jurisdiction"],
            "baseline_score": rec["composite_score"],
            "average_stressed_score": res["average_stressed_score"],
            "worst_score": res["worst_score"],
            "resilience_class": res["resilience_class"],
            "traceability_pct": trace["traceability_completeness_pct"],
            "minimum_claim_confidence": trace["minimum_claim_confidence"],
            "unresolved_tensions": trace["unresolved_contradictions"],
            "committee_route": route,
            "diligence_tier": diligence_tier,
            "implementation_authorized": False,
            "synthetic": True,
        })

        base_conditions = [
            ("RECONCILE_TENSIONS", "Reconcile each supporting claim against its limiting claim with human-reviewed evidence."),
            ("SUBSTANCE_EVIDENCE", "Test people, premises, decision rights, and operating capability in the proposed hub."),
            ("CALCULATION_REVIEW", "Reperform cash-tax, withholding, transfer-pricing, and minimum-tax calculations."),
            ("DOCUMENTATION_MAP", "Map required contemporaneous documentation, owners, dependencies, and refresh dates."),
        ]
        if diligence_tier == "ENHANCED":
            base_conditions.extend([
                ("MITIGATION_PLAN", "Name an owner, deadline, evidence standard, and failure consequence for each mitigation."),
                ("ALTERNATIVE_HUB", "Evaluate at least one alternative hub and preserve the comparison in the knowledge base."),
            ])
        elif diligence_tier == "EXCLUDED":
            base_conditions.extend([
                ("REDESIGN_OPERATING_MODEL", "Redesign the operating model around demonstrable functions rather than rate outcomes."),
                ("ALTERNATIVE_HUBS", "Evaluate at least two alternative hubs with stronger substance compatibility."),
                ("REBASELINE_SCENARIOS", "Rerun all eight Step 2 scenarios before any return to committee."),
            ])
        for sequence, (condition_type, condition_text) in enumerate(base_conditions, 1):
            condition_rows.append({
                "condition_id": f"COND-{index:03d}-{sequence:02d}",
                "committee_decision_id": committee_decision_id,
                "recommendation_id": rec_id,
                "conglomerate_id": rec["conglomerate_id"],
                "condition_type": condition_type,
                "condition_text": condition_text,
                "owner_role": "Synthetic Tax Diligence Lead" if diligence_tier != "EXCLUDED" else "Synthetic Structure Redesign Lead",
                "status": "OPEN",
                "required_before": "STEP_5_CLOSE" if diligence_tier != "EXCLUDED" else "RETURN_TO_COMMITTEE",
                "synthetic": True,
            })

        exhibit_rows.extend([
            {"exhibit_id": f"EX-{index:03d}-A", "recommendation_id": rec_id, "exhibit_type": "RESILIENCE", "record_count": 8, "vault_reference": "16_Data/stress_test_results.csv", "synthetic": True},
            {"exhibit_id": f"EX-{index:03d}-B", "recommendation_id": rec_id, "exhibit_type": "TRACEABILITY", "record_count": 7, "vault_reference": "16_Data/source_claim_recommendation_trace.csv", "synthetic": True},
            {"exhibit_id": f"EX-{index:03d}-C", "recommendation_id": rec_id, "exhibit_type": "TENSIONS", "record_count": 4, "vault_reference": "16_Data/contradictions.csv", "synthetic": True},
        ])

        rec["status"] = "STEP_4_COMMITTEE_ROUTED"
        rec["step4_committee_version"] = STEP4_COMMITTEE_VERSION
        rec["step4_committee_decision_id"] = committee_decision_id
        rec["step4_committee_route"] = route
        rec["step4_diligence_tier"] = diligence_tier
        rec["implementation_authorized"] = False

        rec_path = output_vault / "09_Recommendations" / f"{rec_id}.md"
        rec_text = rec_path.read_text(encoding="utf-8")
        rec_text += f"""

## Step 4 tax-committee routing

- Committee decision: [[{committee_decision_id}]].
- Route: **{route}**.
- Permitted internal action: **{permitted_action}**.
- Diligence tier: **{diligence_tier}**.
- Recommendation approved: **NO**.
- Implementation authorized: **NO**.

The committee product authorizes only the next internal synthetic work package. It does not approve the proposed structure or any real-world act.
"""
        write_text(rec_path, rec_text)

        write_text(output_vault / "10_Decisions" / f"{committee_decision_id}.md", f"""
---
object_type: committee_decision
committee_decision_id: {committee_decision_id}
prior_decision_gate_id: {rec['decision_id']}
recommendation_id: {rec_id}
conglomerate_id: {rec['conglomerate_id']}
committee_version: {STEP4_COMMITTEE_VERSION}
committee_route: {route}
recommendation_approved: false
implementation_authorized: false
synthetic: true
---

# {committee_decision_id} - Synthetic Tax Committee Decision

## Portfolio item

- Conglomerate: **{rec['conglomerate']}**.
- Recommendation: [[{rec_id}]].
- Candidate hub: **{rec['selected_hub_jurisdiction']}**.
- Step 2 resilience: **{res['resilience_class']}**; average stressed score {res['average_stressed_score']}.
- Step 3 traceability: **{trace['traceability_completeness_pct']}%**; unresolved tensions {trace['unresolved_contradictions']}.

## Bounded decision

- Committee route: **{route}**.
- Permitted internal action: **{permitted_action}**.
- Diligence tier: **{diligence_tier}**.
- Rationale: {rationale}

## Hard boundary

The proposed structure is not approved. No filing position, transaction, communication, restructuring, adviser instruction, or implementation is authorized.
""")

    for group in conglomerates:
        rec_id = next(r["recommendation_id"] for r in recommendations if r["conglomerate_id"] == group["conglomerate_id"])
        decision = decision_by_rec[rec_id]
        group["recommendation_status"] = decision["committee_route"]
        group["step4_committee_decision_id"] = decision["committee_decision_id"]

    write_csv(data / "committee_decisions.csv", decision_rows)
    write_csv(data / "committee_portfolio.csv", portfolio_rows)
    write_csv(data / "committee_conditions.csv", condition_rows)
    write_csv(data / "committee_exhibits.csv", exhibit_rows)
    write_csv(data / "recommendations.csv", recommendations)
    write_csv(data / "conglomerates.csv", conglomerates)

    route_counts = Counter(row["committee_route"] for row in decision_rows)
    tier_counts = Counter(row["diligence_tier"] for row in decision_rows)
    portfolio_sorted = sorted(portfolio_rows, key=lambda r: float(r["average_stressed_score"]), reverse=True)

    decision_table = md_table(decision_rows, [
        ("committee_decision_id", "Decision"), ("conglomerate", "Conglomerate"),
        ("committee_route", "Route"), ("diligence_tier", "Tier"),
        ("recommendation_approved", "Approved"), ("implementation_authorized", "Implementation"),
    ])
    portfolio_table = md_table(portfolio_sorted, [
        ("conglomerate", "Conglomerate"), ("candidate_hub", "Hub"),
        ("baseline_score", "Baseline"), ("average_stressed_score", "Stressed"),
        ("worst_score", "Worst"), ("resilience_class", "Resilience"),
        ("traceability_pct", "Traceability"), ("committee_route", "Route"),
    ])
    condition_summary = defaultdict(int)
    for row in condition_rows:
        condition_summary[row["recommendation_id"]] += 1
    condition_table = md_table([
        {
            "recommendation_id": row["recommendation_id"],
            "conglomerate": row["conglomerate"],
            "conditions": condition_summary[row["recommendation_id"]],
            "tier": row["diligence_tier"],
            "route": row["committee_route"],
        } for row in portfolio_rows
    ], [("recommendation_id", "Recommendation"), ("conglomerate", "Conglomerate"),
        ("conditions", "Open conditions"), ("tier", "Tier"), ("route", "Route")])

    memo = f"""
---
object_type: tax_committee_memorandum
report_id: RPT-STEP-4-MEMO
quarter: {STEP4_QUARTER}
committee_version: {STEP4_COMMITTEE_VERSION}
status: INTERNAL_SYNTHETIC_COMMITTEE_PRODUCT
synthetic: true
---

# Tax Planning Exo-Brain - First Tax Committee Memorandum

**Committee cycle:** {STEP4_QUARTER}  
**Committee version:** {STEP4_COMMITTEE_VERSION}  
**Decision requested:** approve the bounded routing of ten synthetic Recommendation V1 records into the next analytical stage; do not approve any structure or implementation.

## Executive conclusion

The committee can responsibly advance nine portfolio items into controlled synthetic diligence. Four robust, fully traceable recommendations may enter standard Step 5 diligence. Five fully traceable recommendations with managed fragility may enter enhanced diligence only with mitigation plans and named exception ownership. Cobalt Life Sciences remains on hold for substance-led redesign and is excluded from the Step 5 diligence set.

The portfolio is not implementation-ready. Every recommendation retains four unresolved tensions, and the weakest portfolio-wide scenario remains reduced operating substance. Complete provenance permits informed scrutiny; it does not convert an analytical design into tax advice or authority.

## Portfolio decision

- **4** items: advance to bounded standard diligence.
- **5** items: advance to enhanced diligence with mitigation controls.
- **1** item: hold for redesign and rerun.
- **0** structures approved.
- **0** real-world actions authorized.

## Committee routing register

{decision_table}

## Evidence considered

The committee product relies on the frozen Step 3 record: 44 immutable source snapshots, 70 atomic claims, 40 active dependencies, 40 unresolved tensions, and 70 complete source-to-decision trace rows. Step 2 contributes 80 controlled scenario evaluations. The 100 synthetic tax-code modules and ten-company universe remain unchanged.

## Principal risk conclusion

Operating substance is the dominant cross-portfolio fragility. Accordingly, Step 5 must test actual functions, people, premises, decision rights, and documentary capability before any economic optimization. Managed-fragility cases also require a documented alternative-hub comparison. Cobalt must be redesigned around demonstrable operating functions before it may return to committee.

## Governance boundary

This is an internal synthetic committee product. The only decisions available are to commission further synthetic diligence, impose mitigations, request redesign, or hold a recommendation. The memorandum does not authorize a filing position, transaction, communication, restructuring, adviser instruction, or implementation.
"""
    dashboard = f"""
---
object_type: committee_dashboard
report_id: RPT-STEP-4-DASH
quarter: {STEP4_QUARTER}
committee_version: {STEP4_COMMITTEE_VERSION}
synthetic: true
---

# Step 4 Tax Committee Dashboard

## Portfolio KPIs

| KPI | Value |
|---|---:|
| Recommendations reviewed | 10 |
| Standard diligence | {tier_counts['STANDARD']} |
| Enhanced diligence | {tier_counts['ENHANCED']} |
| Excluded / redesign | {tier_counts['EXCLUDED']} |
| Traceability complete | 10 |
| Unresolved tensions | {len(contradictions)} |
| Open committee conditions | {len(condition_rows)} |
| Structures approved | 0 |
| Implementation authorized | 0 |

## Ranked portfolio

{portfolio_table}

## Committee interpretation

The stressed score ranks analytical resilience; it is not a tax-savings estimate and cannot authorize a structure. Every item retains explicit unresolved tensions.
"""
    exhibits = f"""
---
object_type: committee_exhibits
report_id: RPT-STEP-4-EXHIBITS
quarter: {STEP4_QUARTER}
committee_version: {STEP4_COMMITTEE_VERSION}
synthetic: true
---

# Step 4 Committee Exhibits

## Exhibit A - Portfolio resilience and routing

{portfolio_table}

## Exhibit B - Conditions by recommendation

{condition_table}

## Exhibit C - Evidence architecture

- 100 synthetic tax-code modules in the governed inventory.
- 44 frozen relied-upon source snapshots.
- 70 atomic claims: 30 supporting and 40 limiting.
- 40 unresolved design tensions.
- 80 Step 2 scenario results.
- 30 recommendation-specific committee exhibit references.

## Exhibit D - Permission model

Evidence may narrow reliance. Only a human committee decision may widen the internal analytical task, and no Step 4 decision may authorize implementation.
"""

    packet_dir = output_vault / "15_Committee_Packets"
    reports_dir = output_vault / "12_Reports"
    write_text(packet_dir / "TAX_COMMITTEE_MEMORANDUM.md", memo)
    write_text(packet_dir / "COMMITTEE_DASHBOARD.md", dashboard)
    write_text(packet_dir / "COMMITTEE_EXHIBITS.md", exhibits)
    write_text(packet_dir / "COMMITTEE_DECISION_LOG.md", "# Step 4 Committee Decision Log\n\n" + decision_table)
    write_text(reports_dir / "STEP_4_TAX_COMMITTEE_MEMORANDUM.md", memo)
    write_text(reports_dir / "STEP_4_COMMITTEE_DASHBOARD.md", dashboard)
    write_text(reports_dir / "STEP_4_COMMITTEE_EXHIBITS.md", exhibits)

    write_text(output_vault / "11_Quarterly_Updates" / STEP4_QUARTER / "STEP_4_COMMITTEE_CYCLE.md", f"""
# {STEP4_QUARTER} - Step 4 Committee Cycle

- Committee decisions: {len(decision_rows)}.
- Standard diligence authorizations: {tier_counts['STANDARD']}.
- Enhanced diligence authorizations: {tier_counts['ENHANCED']}.
- Holds for redesign: {tier_counts['EXCLUDED']}.
- Open committee conditions: {len(condition_rows)}.
- Tax-code alterations: 0.
- New companies: 0.
- Structures approved: 0.
- Implementation authority: none.
""")

    write_text(output_vault / "14_Hot_Cache" / "CURRENT_STATE.md", f"""
# Current State - {STEP4_QUARTER} Tax Committee Product

- Active step: 4 of 10
- Tax-code modules: 100 (unchanged)
- Conglomerates: 10 (unchanged)
- Committee decisions: 10
- Standard Step 5 diligence: {tier_counts['STANDARD']}
- Enhanced Step 5 diligence: {tier_counts['ENHANCED']}
- Excluded pending redesign: {tier_counts['EXCLUDED']}
- Open committee conditions: {len(condition_rows)}
- Unresolved tensions: {len(contradictions)}
- Structures approved: 0
- Explicit prohibition: no real tax advice, filing, transaction, communication, adviser instruction, or restructuring
""")

    state = {
        "project": "Tax Planning Exo-Brain",
        "active_step": 4,
        "quarter": STEP4_QUARTER,
        "recommendation_version": "2026-Q3-R001",
        "stress_test_version": "2026-Q3-S001",
        "provenance_version": "2026-Q3-P001",
        "committee_version": STEP4_COMMITTEE_VERSION,
        "counts": {
            "jurisdictions": 20,
            "tax_codes": len(tax_codes),
            "conglomerates": len(conglomerates),
            "entities": len(entities),
            "transactions": len(transactions),
            "recommendations": len(recommendations),
            "relied_upon_source_snapshots": len(sources),
            "atomic_claims": len(claims),
            "contradictions": len(contradictions),
            "committee_decisions": len(decision_rows),
            "committee_conditions": len(condition_rows),
            "committee_exhibits": len(exhibit_rows),
        },
        "committee_routes": dict(route_counts),
        "diligence_tiers": dict(tier_counts),
        "recommendation_approved": False,
        "implementation_authorized": False,
        "permission": "BOUNDED_INTERNAL_DECISIONS_ONLY",
        "synthetic": True,
    }
    write_text(output_vault / "00_System" / "CURRENT_STATE.json", json.dumps(state, indent=2))

    excluded = {
        "13_Audit/STEP_4_VALIDATION.json",
        "13_Audit/STEP_4_MANIFEST.csv",
        "13_Audit/STEP_4_SUCCESS.md",
    }
    manifest_rows = []
    for path in sorted(p for p in output_vault.rglob("*") if p.is_file()):
        rel = path.relative_to(output_vault).as_posix()
        if rel in excluded:
            continue
        manifest_rows.append({"path": rel, "bytes": path.stat().st_size, "sha256": sha256(path)})
    write_csv(output_vault / "13_Audit" / "STEP_4_MANIFEST.csv", manifest_rows)

    checks = {
        "step3_success_marker_present": (output_vault / "13_Audit" / "STEP_3_SUCCESS.md").exists(),
        "tax_codes_equal_100": len(tax_codes) == 100,
        "conglomerates_equal_10": len(conglomerates) == 10,
        "recommendations_equal_10": len(recommendations) == 10,
        "source_snapshots_equal_44": len(sources) == 44,
        "atomic_claims_equal_70": len(claims) == 70,
        "contradictions_equal_40": len(contradictions) == 40,
        "all_contradictions_remain_unresolved": all(r["status"] == "UNRESOLVED" for r in contradictions),
        "committee_decisions_equal_10": len(decision_rows) == 10,
        "standard_diligence_equal_4": tier_counts["STANDARD"] == 4,
        "enhanced_diligence_equal_5": tier_counts["ENHANCED"] == 5,
        "excluded_equal_1": tier_counts["EXCLUDED"] == 1,
        "cobalt_is_excluded": decision_by_rec["REC-003-V001"]["committee_route"] == "HOLD_FOR_REDESIGN",
        "committee_conditions_equal_53": len(condition_rows) == 53,
        "committee_exhibits_equal_30": len(exhibit_rows) == 30,
        "no_recommendation_approved": not any(bool_text(r["recommendation_approved"]) == "true" for r in decision_rows),
        "implementation_never_authorized": not any(bool_text(r["implementation_authorized"]) == "true" for r in decision_rows),
        "no_tax_code_versions_changed": all(r["version"] == "2026-Q3-V001" for r in tax_codes),
        "no_new_companies": len(conglomerates) == 10,
        "committee_product_files_present": all((packet_dir / name).exists() for name in [
            "TAX_COMMITTEE_MEMORANDUM.md", "COMMITTEE_DASHBOARD.md", "COMMITTEE_EXHIBITS.md", "COMMITTEE_DECISION_LOG.md"
        ]),
    }
    validation = {
        "step": 4,
        "date": STEP4_DATE,
        "quarter": STEP4_QUARTER,
        "committee_version": STEP4_COMMITTEE_VERSION,
        "checks": checks,
        "counts": state["counts"],
        "committee_routes": dict(route_counts),
        "diligence_tiers": dict(tier_counts),
        "status": "PASS" if all(checks.values()) else "FAIL",
    }
    write_text(output_vault / "13_Audit" / "STEP_4_VALIDATION.json", json.dumps(validation, indent=2))
    if validation["status"] != "PASS":
        failed = [key for key, ok in checks.items() if not ok]
        raise AssertionError("Step 4 validation failed: " + ", ".join(failed))
    write_text(output_vault / "13_Audit" / "STEP_4_SUCCESS.md", f"""
# Step 4 Validation: PASS

Validated 10 bounded committee decisions: 4 standard diligence, 5 enhanced diligence with mitigations, and 1 hold for redesign. The portfolio contains {len(condition_rows)} open committee conditions and {len(exhibit_rows)} exhibit links. No structure is approved and no implementation is authorized.
""")
    return validation


In [ ]:
PROJECT = Path('/content/drive/MyDrive/Tax_Planning_ExoBrain_Project')
SOURCE_VAULT = PROJECT / 'Step_3' / 'Tax_Planning_ExoBrain_Vault'
OUTPUT_VAULT = PROJECT / 'Step_4' / 'Tax_Planning_ExoBrain_Vault'
validation = apply_step4(SOURCE_VAULT, OUTPUT_VAULT)
print(json.dumps(validation, indent=2))
print(f'\nStep 4 vault created at: {OUTPUT_VAULT}')


## Expected result

Validation must be `PASS`: 10 committee decisions, 4 standard diligence routes, 5 enhanced diligence routes, 1 hold for redesign, 53 open committee conditions, and 30 exhibit links. The 100 tax codes, 10 conglomerates, 44 source snapshots, 70 claims, and 40 unresolved tensions remain intact. No structure is approved and no implementation is authorized.
